In [1]:
from jinja2 import Template
import yaml

with open("../workloads/gpt3_175B.yaml") as f:
    text = f.read()

template = Template(text)

# # You can override these if you want
rendered = template.render()

workload = yaml.safe_load(rendered)

einsums = workload["workload"]["einsums"]
rank_sizes = workload["workload"]["rank_sizes"]

print(einsums)

[{'einsum': 'I[b, m, d] = I_in[b, m, d]', 'is_copy_operation': True}, 'V[b, m, h, e] = I[b, m, d] * WV[h, e, d]', 'K[b, m, h, e] = I[b, m, d] * WK[h, e, d]', 'Q[b, m, h, e] = I[b, m, d] * WQ[h, e, d]', {'einsum': 'QK[b, m, p, h] = Q[b, m, h, e] * K[b, M: p, h, e]', 'renames': {'input': 'Q'}}, 'QK_softmax[b, m, p, h] = QK[b, m, p, h]', {'einsum': 'AV[b, m, h, f] = QK_softmax[b, m, p, h] * V[b, M: p, h, E: f]', 'renames': {'input': 'QK_softmax'}}, 'Z[b, m, g] = AV[b, m, h, f] * WZ[h, f, g]', 'FFA[b, m, c] = Z[b, m, g] * WFFA[g, c]', 'FFB[b, m, j] = FFA[b, m, c] * WFFB[c, j]']


In [5]:
import torch
B = 1
M = 128
P = 128
H = 8
E = 32
F = 32
D = H * E
G = H * E
C = 4 * G
J = G

I = torch.randn(B, M, D)

WV = torch.randn(H, E, D)
WK = torch.randn(H, E, D)
WQ = torch.randn(H, E, D)

WFFA = torch.randn(G, C)
WFFB = torch.randn(C, J)

In [9]:
def run_model(I, WV, WK, WQ, WZ, WFFA, WFFB):
    V = torch.einsum("bmd,hed->bmhe", I, WV)
    K = torch.einsum("bmd,hed->bmhe", I, WK)
    Q = torch.einsum("bmd,hed->bmhe", I, WQ)

    QK = torch.einsum("bmhe,bphe->bmph", Q, K)
    QK_softmax = torch.softmax(QK, dim=2)

    AV = torch.einsum("bmph,bphe->bmhe", QK_softmax, V)

    Z = torch.einsum("bmhe,heg->bmg", AV, WZ)
    FFA = torch.einsum("bmg,gc->bmc", Z, WFFA)
    FFB = torch.einsum("bmc,cj->bmj", FFA, WFFB)

    return FFB

In [18]:
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=True
) as prof:
    
    out = run_model(I, WV, WK, WQ, WV, WFFA, WFFB)

print(prof.key_averages().table(sort_by="cpu_time_total"))

------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                    Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
            aten::einsum         4.64%     857.682us        75.18%      13.900ms       1.738ms             8  
               aten::bmm        57.27%      10.589ms        60.19%      11.128ms       1.391ms             8  
           aten::softmax         0.26%      47.388us        24.82%       4.588ms       4.588ms             1  
          aten::_softmax        15.43%       2.852ms        24.56%       4.541ms       4.541ms             1  
             aten::clone         1.02%     188.422us        11.42%       2.112ms     191.958us            11  
             aten::copy_         9.31%       1.722ms         9.31%       1.722ms     156.528us            11  
 